# 03 - DTW consensus state clustering

Thin caller over `nfip.dtw_clustering`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from collections import defaultdict
from scipy.cluster.hierarchy import dendrogram
from sklearn.manifold import MDS
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
!pip install tslearn
from src import data, preprocess, config
from src.dtw_clustering import (run_simulation_clustering, cluster_records_to_wide,
    wide_to_cluster_records, build_consensus_clusters)

## Params

In [ ]:
optimal_cluster = 'st_cluster_3_5_7'
save = True
simulate = True
simulation_type = 'block'

## Load geospatial + clustered claims (CPI-adjusted, event-relabeled)

In [ ]:
gdf_counties = data.load_counties()
gdf_states   = data.load_states()

In [ ]:
clustered_claims, optimal_cluster = data.load_clustered_claims('')  # sensitivity CSV
clustered_claims = preprocess.add_claim_fields(clustered_claims)
_cpi = data.load_cpi_annual()
clustered_claims = preprocess.cpi_adjust_claims(clustered_claims, _cpi)
clustered_claims = preprocess.relabel_events(clustered_claims, optimal_cluster)

## Load per-simulation state balances

In [ ]:
nfip_balances = pd.read_csv('Results/state_balance_' + simulation_type + '_new.csv')
nfip_balances['STATEFP'] = nfip_balances['STATEFP'].astype(str).str.zfill(2)
nfip_balances['state_abbrev'] = nfip_balances['STATEFP'].map(config.FIPS_TO_ABBREV)

## Cluster each simulation, then build consensus

In [ ]:
if simulate:
    cluster_records = run_simulation_clustering(nfip_balances, k=4)
    df_wide = cluster_records_to_wide(cluster_records)
    df_wide.to_csv('Results/DTW_HC_simulation_clusters' + simulation_type + '.csv', index=False)
else:
    df_wide = pd.read_csv('Results/DTW_HC_simulation_clusters4.csv')
    cluster_records = wide_to_cluster_records(df_wide)

In [ ]:
cluster_df, Z, coassoc_matrix = build_consensus_clusters(cluster_records, k=3)
states = sorted(cluster_records.keys())

## Final composite panel (verbatim)

In [ ]:
custom_cmap = ListedColormap(
    ["#F4EEB5", "#C96A55", "#6299C3"],
    name="custom_ybr"
)

In [ ]:
# Ensure column name is consistent for merge
cluster_df = cluster_df.rename(columns={"consensus_cluster": "cluster"})

# Merge into GeoDataFrame
gdf_plot = gdf_states.copy()
gdf_plot = gdf_plot.merge(cluster_df, left_on="STUSPS", right_on="state", how="left")

# MDS projection
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=0)
coords = mds.fit_transform(1 - coassoc_matrix)

mds_df = pd.DataFrame(coords, columns=["MDS1", "MDS2"])
mds_df["cluster"] = cluster_df["cluster"].values
mds_df["label"] = cluster_df["state"].values

# Set up figure layout
fig = plt.figure(figsize=(12, 8))
gs = gridspec.GridSpec(2, 3, height_ratios=[3, 2], width_ratios=[1, 1, 1])

# panel (a): Map
ax_map = fig.add_subplot(gs[0, 0:2])
map_plot = gdf_plot.plot(
    column="cluster", cmap=custom_cmap, ax=ax_map, edgecolor="black", linewidth=0.5,
    legend=False
)
ax_map.set_title("US States Colored by Cluster")
ax_map.axis("off")
ax_map.text(-0.1, 1.05, "(a)", transform=ax_map.transAxes, fontsize=12, fontweight='bold')

# Manual legend
unique_clusters = sorted(gdf_plot["cluster"].dropna().unique())
cmap = plt.cm.get_cmap(custom_cmap, len(unique_clusters))
legend_elements = [
    Patch(facecolor=cmap(i), edgecolor='black', label=f'Cluster {int(c)}')
    for i, c in enumerate(unique_clusters)
]
ax_map.legend(handles=legend_elements, loc="lower right")

# panel (b): MDS projection
ax_mds = fig.add_subplot(gs[0, 2])
scatter = ax_mds.scatter(mds_df["MDS1"], mds_df["MDS2"], c=mds_df["cluster"], cmap=custom_cmap)
for _, row in mds_df.iterrows():
    ax_mds.text(row["MDS1"], row["MDS2"], row["label"], fontsize=7, alpha=0.7)
ax_mds.set_title("MDS Projection")
ax_mds.spines[['top', 'right']].set_visible(False)
ax_mds.set_xlabel("MDS Dimension 1")
ax_mds.set_ylabel("MDS Dimension 2")
ax_mds.text(-0.2, 1.05, "(b)", transform=ax_mds.transAxes, fontsize=12, fontweight='bold')

# panel (c): Dendrogram
ax_dendro = fig.add_subplot(gs[1, :])
d = dendrogram(Z, labels=states, leaf_rotation=90, ax=ax_dendro, color_threshold=0)

# Make all dendrogram lines black and thin
for icoord, dcoord in zip(d['icoord'], d['dcoord']):
    ax_dendro.plot(icoord, dcoord, color='black', linewidth=0.5)

ax_dendro.set_title("Dendrogram")
ax_dendro.spines[['top', 'right', 'left']].set_visible(False)
ax_dendro.yaxis.set_ticks([])
ax_dendro.set_yticklabels([])
ax_dendro.text(-0.05, 1.05, "(c)", transform=ax_dendro.transAxes, fontsize=12, fontweight='bold')

plt.tight_layout()

plt.show()